# llm-request SDK 종합 검증 노트북 (`verify_all.ipynb`)

`llm-request` SDK의 **모든 공개 함수 × 모든 분기**를 로컬 환경에서 실행·검증한다.

## 선행 조건

다음이 모두 실행 중이어야 한다 (localhost 기준):

| 서비스 | 포트 | 실행 방법 |
|--------|------|----------|
| Redis | 6379 | `docker compose -f infra/docker-compose.yml up -d redis` |
| MongoDB | 27017 | `docker compose -f infra/docker-compose.yml up -d mongodb` |
| llm-handler | 8100 | `docker compose --profile handlers up -d llm-handler` |
| mdb-handler | 8300 | `docker compose --profile handlers up -d mdb-handler` |
| file-manager | 8102 | `cd packages/file-manager/service && uv run uvicorn app.main:app --port 8102` |
| llm-request | 8101 | `cd packages/llm-request/service && uv run uvicorn app.main:app --port 8101` |

`.env`에 `GEMINI_API_KEY` 필수.

## 검증 범위

- 공개 함수 6종: `llm_request`, `llm_request_async`, `llm_request_stream`, `llm_request_stream_async`, `done_check`, `fetch_by_cache_key`
- 플래그: `cache_mode`, `save_mode`, `debug`, `ttl_seconds`, `context_cache_ttl`
- 파일 분기: inline 단일/복수, Files API, 부분 섞기, 20MB 한도, PDF split, context_cache
- 오류 경로: 지원 안 되는 mime 타입
- 동시성: single-flight dedup

## 0. 환경 설정

In [1]:
import os

os.environ["LLM_REQUEST_URL"]  = "http://localhost:8101"
os.environ["FILE_MANAGER_URL"] = "http://localhost:8102"
os.environ["MONGODB_DATABASE"] = "kw_ai_dev"  # 테스트 전용 DB

TEST_DB = os.environ["MONGODB_DATABASE"]
print(f"LLM_REQUEST_URL  = {os.environ['LLM_REQUEST_URL']}")
print(f"FILE_MANAGER_URL = {os.environ['FILE_MANAGER_URL']}")
print(f"MONGODB_DATABASE = {TEST_DB}")

LLM_REQUEST_URL  = http://localhost:8101
FILE_MANAGER_URL = http://localhost:8102
MONGODB_DATABASE = kw_ai_dev


## 1. 공통 유틸리티

In [2]:
import time
import uuid
from contextlib import contextmanager
from pathlib import Path


@contextmanager
def timer(label: str):
    start = time.perf_counter()
    yield
    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"  [{label}] {elapsed_ms:.0f}ms")


def print_meta(title: str, meta: dict) -> None:
    print(f"--- {title} ---")
    print(f"  source        = {meta.get('source')}")
    print(f"  cache_hit     = {meta.get('cache_hit')}")
    print(f"  cache_key     = {str(meta.get('cache_key',''))[:16]}...")
    if meta.get('route_reason'):
        print(f"  route_reason  = {meta['route_reason']}")
    if meta.get('file_route'):
        print(f"  file_route    = {meta['file_route']}")


def unique_q(base: str) -> str:
    """캐시 미스를 강제하기 위해 매번 다른 입력 생성."""
    return f"{base} [id={uuid.uuid4().hex[:8]}]"

In [3]:
import sys; print(sys.executable)


/Users/yeonsoo-kim/01.kiwoom/01.github/Kiwoom-Rag/packages/llm-handler/service/.venv/bin/python


In [5]:
from file_manager import FileMeta, acheck_cache, aupload_file


async def ensure_file_meta(path) -> FileMeta:
    """file-manager에 파일 등록 후 FileMeta 반환. check_cache → miss면 upload."""
    p = Path(path)
    stat = p.stat()
    metadata = {
        "file_name": p.name,
        "file_mtime": int(stat.st_mtime),
        "file_size": stat.st_size,
    }
    hit = await acheck_cache(metadata)
    if hit.hit and hit.metadata is not None:
        return hit.metadata
    data = p.read_bytes()
    return await aupload_file(data, metadata)

ModuleNotFoundError: No module named 'file_manager'

## 2. 서비스 헬스체크

모든 서비스가 올라와 있는지 확인. 하나라도 FAIL이면 이후 셀은 실패.

In [ ]:
import httpx

SERVICES = [
    ("llm-request",  "http://localhost:8101/health"),
    ("file-manager", "http://localhost:8102/health"),
    ("llm-handler",  "http://localhost:8100/health"),
    ("mdb-handler",  "http://localhost:8300/health"),
]

async with httpx.AsyncClient(timeout=5.0) as c:
    for name, url in SERVICES:
        try:
            r = await c.get(url)
            ok = r.status_code == 200
            print(f"  [{'OK' if ok else 'FAIL'}] {name:<13} -> {r.status_code}")
        except Exception as e:
            print(f"  [FAIL] {name:<13} -> {type(e).__name__}: {e}")

## 3. `llm_request_async` — 기본 (캐시 miss → hit)

동일 입력 2회 호출해 첫 번째는 LLM, 두 번째는 MDB 캐시 히트가 되는지 확인.

In [ ]:
from llm_request import llm_request_async

Q1 = unique_q("대한민국의 수도는?")

with timer("1회차 (cache miss → LLM)"):
    r1, m1 = await llm_request_async(
        {"contents": Q1, "model": "gemini-2.5-flash"},
        database=TEST_DB,
    )
print_meta("1회차", m1)
print(f"  text          = {r1.get('text','')[:80]}")

with timer("2회차 (cache hit)"):
    r2, m2 = await llm_request_async(
        {"contents": Q1, "model": "gemini-2.5-flash"},
        database=TEST_DB,
    )
print_meta("2회차", m2)

assert m2.get("cache_hit") is True, "2회차는 캐시 히트여야 한다"
assert m1.get("cache_key") == m2.get("cache_key"), "동일 입력 → 동일 cache_key"
print("\n  OK: miss→hit 동작 확인")

## 4. `llm_request` (동기 래퍼)

내부에서 `asyncio.run`을 호출하므로 Jupyter 이벤트 루프 안에서 바로 실행하면 충돌한다. 별도 스레드에서 실행.

In [ ]:
import concurrent.futures
from llm_request import llm_request


def _sync_call():
    return llm_request(
        {"contents": unique_q("2+2는?"), "model": "gemini-2.5-flash"},
        database=TEST_DB,
    )


with concurrent.futures.ThreadPoolExecutor() as ex:
    r, m = ex.submit(_sync_call).result()

print_meta("sync", m)
print(f"  text          = {r.get('text','')[:80]}")

## 5. 플래그 — `cache_mode` / `save_mode` / `debug` / `ttl_seconds`

- `cache_mode=False`: 캐시 조회 스킵 → 항상 LLM 호출
- `save_mode=False`: 응답 저장 안 함 → 이후 done_check 시 miss
- `debug=True`: meta에 `route_reason` 포함
- `ttl_seconds`: MDB TTL 지정

In [ ]:
# cache_mode=False : 첫 호출로 저장된 결과가 있어도 무시하고 LLM 호출
Q_CM = unique_q("캐시 우회 테스트")
await llm_request_async({"contents": Q_CM, "model": "gemini-2.5-flash"}, database=TEST_DB)
r, m = await llm_request_async(
    {"contents": Q_CM, "model": "gemini-2.5-flash"},
    cache_mode=False,
    database=TEST_DB,
)
print_meta("cache_mode=False (두번째 호출도 LLM)", m)
assert m.get("cache_hit") is False
print("  OK: cache_mode=False는 캐시 히트 안 됨")

In [ ]:
# save_mode=False : 호출은 하지만 저장은 안 함 → done_check 시 miss
from llm_request import done_check

Q_SM = unique_q("저장 스킵 테스트")
r, m = await llm_request_async(
    {"contents": Q_SM, "model": "gemini-2.5-flash"},
    save_mode=False,
    database=TEST_DB,
)
print_meta("save_mode=False", m)

dc = await done_check({"contents": Q_SM, "model": "gemini-2.5-flash"}, database=TEST_DB)
print(f"  done_check.cache_hit = {dc['cache_hit']}")
assert dc["cache_hit"] is False, "save_mode=False 이후 done_check는 miss여야 한다"
print("  OK: save_mode=False는 저장 안 함")

In [ ]:
# debug=True : meta.route_reason 확인
r, m = await llm_request_async(
    {"contents": unique_q("debug 경로"), "model": "gemini-2.5-flash"},
    debug=True,
    database=TEST_DB,
)
print_meta("debug=True", m)
print(f"  route_reason  = {m.get('route_reason')!r}")

In [ ]:
# ttl_seconds : MDB TTL 지정 (저장은 되지만 주기적으로 만료)
r, m = await llm_request_async(
    {"contents": unique_q("TTL 테스트"), "model": "gemini-2.5-flash"},
    ttl_seconds=120,
    database=TEST_DB,
)
print_meta("ttl_seconds=120", m)

## 6. `done_check` + `fetch_by_cache_key`

In [ ]:
from llm_request import fetch_by_cache_key

Q_F = unique_q("페치 테스트 질문")
await llm_request_async({"contents": Q_F, "model": "gemini-2.5-flash"}, database=TEST_DB)

dc = await done_check({"contents": Q_F, "model": "gemini-2.5-flash"}, database=TEST_DB)
print(f"done_check: cache_key={dc['cache_key'][:16]}... cache_hit={dc['cache_hit']}")
assert dc["cache_hit"] is True

r, m = await fetch_by_cache_key(dc["cache_key"], database=TEST_DB)
print_meta("fetch_by_cache_key", m)
print(f"  text = {r.get('text','')[:80]}")

## 7. `llm_request_stream_async` — 스트리밍

- 1회차 (miss): SSE 청크 여러 개
- 2회차 (hit): 캐시 결과가 단일 청크로 래핑되어 동일 인터페이스로 반환

In [ ]:
from llm_request import llm_request_stream_async

Q_S = unique_q("스트리밍으로 답해줘: 한국 전통 악기 3가지")

chunks_1 = []
async for ev in llm_request_stream_async(
    {"contents": Q_S, "model": "gemini-2.5-flash"},
    database=TEST_DB,
):
    chunks_1.append(ev)

print(f"1회차: {len(chunks_1)}개 청크")
full_text = "".join(c.get("text", "") for c in chunks_1)
print(f"  전체 텍스트: {full_text[:120]}")

chunks_2 = []
async for ev in llm_request_stream_async(
    {"contents": Q_S, "model": "gemini-2.5-flash"},
    database=TEST_DB,
):
    chunks_2.append(ev)

print(f"\n2회차 (cache hit): {len(chunks_2)}개 청크")

## 8. `llm_request_stream` (동기 스트리밍)

In [ ]:
from llm_request import llm_request_stream


def _sync_stream():
    events = []
    for ev in llm_request_stream(
        {"contents": unique_q("3+3은?"), "model": "gemini-2.5-flash"},
        database=TEST_DB,
    ):
        events.append(ev)
    return events


with concurrent.futures.ThreadPoolExecutor() as ex:
    evs = ex.submit(_sync_stream).result()

print(f"동기 스트림: {len(evs)}개 청크")
print(f"  첫 청크: {evs[0]}")

## 9. 파일 준비 — `ensure_file_meta`

테스트 파일들을 file-manager에 등록하고 FileMeta를 캐시.

In [ ]:
SAMPLES_DIR = Path(".").resolve()

FILES = {
    "small_pdf":  SAMPLES_DIR / "small_file.pdf",    # 612KB
    "medium_pdf": SAMPLES_DIR / "medium_file.pdf",   # 24MB
    "large_pdf":  SAMPLES_DIR / "large_file.pdf",    # 92MB
    "image":      SAMPLES_DIR / "test_image.png",    # 1.9MB
    "text":       SAMPLES_DIR / "sample.txt",        # 2KB
    "audio":      SAMPLES_DIR / "test_audio.m4a",    # 4MB (mime 미지원 테스트용)
}

metas: dict[str, FileMeta] = {}
for key, path in FILES.items():
    if not path.exists():
        print(f"  [SKIP] {key}: {path} 없음")
        continue
    m = await ensure_file_meta(path)
    metas[key] = m
    print(f"  [OK]   {key:<10} mime={m.mime_type:<18} size={m.file_size:>10,}B")

## 10. Inline — PDF 단일 (612KB)

PDF 한 개가 50MB 미만이므로 inline 분기.

In [ ]:
r, m = await llm_request_async(
    {"contents": unique_q("이 문서를 한 문장으로 요약해줘"), "model": "gemini-2.5-flash"},
    file_metas=[metas["small_pdf"]],
    database=TEST_DB,
    debug=True,
)
print_meta("small_pdf (inline)", m)
print(f"  text = {r.get('text','')[:150]}")

## 11. Inline — 이미지

In [ ]:
r, m = await llm_request_async(
    {"contents": unique_q("이 이미지에 무엇이 있나?"), "model": "gemini-2.5-flash"},
    file_metas=[metas["image"]],
    database=TEST_DB,
    debug=True,
)
print_meta("image (inline)", m)
print(f"  text = {r.get('text','')[:150]}")

## 12. Inline — 텍스트

In [ ]:
r, m = await llm_request_async(
    {"contents": unique_q("이 텍스트 파일의 주제는?"), "model": "gemini-2.5-flash"},
    file_metas=[metas["text"]],
    database=TEST_DB,
    debug=True,
)
print_meta("sample.txt (inline)", m)
print(f"  text = {r.get('text','')[:150]}")

## 13. Files API — 대용량 PDF (92MB, 자동 split)

PDF 한 개가 50MB를 초과하면 Files API 분기 + PDF split. 첫 호출은 업로드 포함이라 분 단위로 걸릴 수 있음.

In [ ]:
if "large_pdf" not in metas:
    print("  [SKIP] large_file.pdf 없음")
else:
    with timer("large_pdf 첫 호출 (업로드+split+LLM)"):
        r, m = await llm_request_async(
            {"contents": unique_q("이 PDF의 제목 한 줄"), "model": "gemini-2.5-flash", "max_tokens": 100},
            file_metas=[metas["large_pdf"]],
            database=TEST_DB,
            debug=True,
        )
    print_meta("large_pdf (Files API + split)", m)
    print(f"  text = {r.get('text','')[:150]}")

## 14. 부분 섞기 — inline + Files API 혼재

`small_pdf` + `large_pdf` → 작은 파일은 inline, 큰 파일은 Files API로 분류.

In [ ]:
if "large_pdf" not in metas:
    print("  [SKIP] large_file.pdf 없음")
else:
    r, m = await llm_request_async(
        {"contents": unique_q("두 PDF의 공통점 한 줄"), "model": "gemini-2.5-flash"},
        file_metas=[metas["small_pdf"], metas["large_pdf"]],
        database=TEST_DB,
        debug=True,
    )
    print_meta("small + large (혼합)", m)
    print(f"  text = {r.get('text','')[:150]}")

## 15. 이미지+PDF 혼합 — 20MB 한도 분기

이미지/오디오가 포함되면 inline 합계 한도가 20MB로 낮아진다. `image(1.9MB) + medium_pdf(24MB)` → 합계 26MB → 큰 쪽이 Files API로 승급.

In [ ]:
if "medium_pdf" not in metas:
    print("  [SKIP] medium_file.pdf 없음")
else:
    r, m = await llm_request_async(
        {"contents": unique_q("이미지와 PDF를 함께 설명"), "model": "gemini-2.5-flash"},
        file_metas=[metas["image"], metas["medium_pdf"]],
        database=TEST_DB,
        debug=True,
    )
    print_meta("image + medium_pdf (20MB 한도)", m)
    print(f"  text = {r.get('text','')[:150]}")

## 16. `context_cache_ttl` — Context Cache 생성

Files API로 분류된 파일에 TTL(초)을 지정하면 Gemini Context Cache 생성. 같은 파일 조합으로 반복 호출 시 입력 토큰 90% 할인.

In [ ]:
if "large_pdf" not in metas:
    print("  [SKIP] large_file.pdf 없음")
else:
    r, m = await llm_request_async(
        {"contents": unique_q("첫 질문"), "model": "gemini-2.5-flash"},
        file_metas=[metas["large_pdf"]],
        context_cache_ttl=3600,
        database=TEST_DB,
        debug=True,
    )
    print_meta("context_cache 생성", m)

    r2, m2 = await llm_request_async(
        {"contents": unique_q("두번째 질문"), "model": "gemini-2.5-flash"},
        file_metas=[metas["large_pdf"]],
        context_cache_ttl=3600,
        database=TEST_DB,
        debug=True,
    )
    print_meta("context_cache 재사용", m2)

## 17. 검증 에러 — 지원 안 되는 mime 타입

`test_audio.m4a` (audio/mp4 또는 audio/x-m4a) → 화이트리스트 미포함 → HTTP 422.

In [ ]:
from llm_request import ServiceError

if "audio" not in metas:
    print("  [SKIP] test_audio.m4a 없음")
else:
    print(f"  파일 mime: {metas['audio'].mime_type}")
    try:
        r, m = await llm_request_async(
            {"contents": "전사해줘", "model": "gemini-2.5-flash"},
            file_metas=[metas["audio"]],
            database=TEST_DB,
        )
        print("  [FAIL] 예외 발생 안 함")
    except ServiceError as e:
        print(f"  [OK] ServiceError: status={e.status_code} type={e.error_type}")
        print(f"       message: {str(e)[:120]}")

## 18. 동시성 — single-flight dedup

동일 입력 3개를 동시에 호출하면 첫 번째만 LLM 호출, 나머지는 결과를 공유받는다.

In [ ]:
import asyncio

Q_D = unique_q("동시성 테스트 질문")


async def _one(i: int):
    r, m = await llm_request_async(
        {"contents": Q_D, "model": "gemini-2.5-flash"},
        database=TEST_DB,
        debug=True,
    )
    return i, r.get("text", "")[:40], m


with timer("동시 3개 요청"):
    results = await asyncio.gather(_one(1), _one(2), _one(3))

for i, txt, meta in results:
    route = meta.get("route_reason")
    print(f"  #{i}: cache_hit={meta.get('cache_hit')!s:<5} route={route!r:<30} text={txt}")

texts = {txt for _, txt, _ in results}
print(f"\n  고유 응답 수: {len(texts)} (1이면 dedup 동작)")

## 정리

위 셀들이 모두 성공하면 다음이 검증된 것:

- 공개 함수 6종 전부 정상 동작
- 캐시 경로: miss → hit, 우회(cache_mode=False), 저장 스킵(save_mode=False)
- 파일 분기: inline(PDF/이미지/텍스트), Files API, 부분 섞기, 20MB 한도 분기, PDF split
- context_cache 생성/재사용
- 입력 검증(mime) fail-fast
- single-flight dedup

실패한 셀이 있으면 §2 헬스체크부터 재확인.